# 03 — Classical ML baseline
Loads real image-derived features, creates a stratified held-out split, fits scaled Logistic Regression and Random Forest, and evaluates held-out predictions. Run from repository root or adjust `DATA_DIR`. These results come from your dataset and are never prefilled.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.preprocessing import load_feature_dataset
DATA_DIR = Path('data/images')
X, y, paths = load_feature_dataset(DATA_DIR)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
print('X:', X.shape, 'class counts:', np.bincount(y))

In [ ]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
}
rows = []
fig, axes = plt.subplots(1, len(models), figsize=(10, 4))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'model': name, 'accuracy': accuracy_score(y_test, pred),
                 'precision_synthetic': precision_score(y_test, pred, zero_division=0),
                 'recall_synthetic': recall_score(y_test, pred, zero_division=0),
                 'f1_synthetic': f1_score(y_test, pred, zero_division=0)})
    print(name); print(classification_report(y_test, pred, target_names=['real', 'synthetic'], zero_division=0))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=['real', 'synthetic'], yticklabels=['real', 'synthetic'], ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()
comparison = pd.DataFrame(rows).set_index('model'); comparison

## Metric interpretation
Accuracy is the share of all correct predictions and can hide poor minority-class performance. Precision asks: among predicted synthetic samples, how many are actually synthetic? Recall asks: among truly synthetic samples, how many are detected? F1 balances precision and recall. Confusion matrix rows are actual classes and columns are predicted classes (real, synthetic). A single random image split can leak related content and does not establish performance on unseen sources/generators; consider source-grouped splits.